# Camada Gold - Agregações e Análise

Le os dados da camada Silver e cria tabelas agregadas prontas para análise/BI: vendas por categoria, resumo de pedidos, avaliação média por produto e resumo de clientes.

In [0]:
from pyspark.sql.functions import *

# Criar schema gold se não existir
spark.sql("CREATE DATABASE IF NOT EXISTS gold")

# Função para converter string 'R$ 1.981,30' ou '-R$ 50,00' em decimal
def parse_currency_br(col_name):
    is_negative = col(col_name).startswith("-")
    cleaned = regexp_replace(col(col_name), "[R$\\s-]", "")  # remove R$, espacos, -
    cleaned = regexp_replace(cleaned, "\\.", "")               # remove separador de milhar
    cleaned = regexp_replace(cleaned, ",", ".")                  # troca virgula por ponto
    return when(is_negative, (-cleaned.cast("decimal(10,2)"))).otherwise(cleaned.cast("decimal(10,2)"))

# Carregar todas as tabelas da camada Silver
silver_tables = {}
for table_name in ["payments", "customers", "orders", "order_items", "products", "reviews"]:
    silver_tables[table_name] = spark.table(f"silver.{table_name}")
    print(f"✓ silver.{table_name} carregada | {silver_tables[table_name].count()} linhas")

print("\nTabelas Silver carregadas e prontas para agregação.")

In [0]:
# Vendas por categoria de produto
# unit_price e price são DECIMAL na Silver (coluna _br tem o formato R$)
df_vendas_categoria = (
    silver_tables["order_items"]
    .join(silver_tables["products"], on="product_id", how="left")
    .withColumn("receita", col("quantity") * parse_currency_br("unit_price"))
    .groupBy("category")
    .agg(
        sum("receita").alias("receita_total"),
        sum("quantity").alias("itens_vendidos"),
        countDistinct("product_id").alias("produtos_distintos"),
        count("order_id").alias("total_pedidos"),
    )
    .orderBy(col("receita_total").desc())
)
df_vendas_categoria.write.format("delta").mode("overwrite").saveAsTable("gold.vendas_por_categoria")
print("✓ gold.vendas_por_categoria criada")
display(df_vendas_categoria)

In [0]:
# Resumo de pedidos por status
df_pedidos_status = (
    silver_tables["orders"]
    .groupBy("status")
    .agg(
        count("order_id").alias("total_pedidos"),
        sum(parse_currency_br("total_amount")).alias("receita_total"),
        avg(parse_currency_br("total_amount")).alias("ticket_medio"),
    )
    .orderBy(col("receita_total").desc())
)
df_pedidos_status.write.format("delta").mode("overwrite").saveAsTable("gold.pedidos_por_status")
print("✓ gold.pedidos_por_status criada")
display(df_pedidos_status)


In [0]:
# Avaliação média por produto
df_avaliacao_produto = (
    silver_tables["reviews"]
    .join(silver_tables["order_items"], on="order_id", how="left")
    .join(silver_tables["products"], on="product_id", how="left")
    .groupBy("product_id", "product_name", "category")
    .agg(
        avg("rating").alias("avaliacao_media"),
        count("review_id").alias("total_avaliacoes"),
        min("rating").alias("pior_nota"),
        max("rating").alias("melhor_nota"),
    )
    .orderBy(col("avaliacao_media").desc())
)
df_avaliacao_produto.write.format("delta").mode("overwrite").saveAsTable("gold.avaliacao_produto")
print("✓ gold.avaliacao_produto criada")
display(df_avaliacao_produto)

In [0]:
# Resumo de clientes
df_resumo_clientes = (
    silver_tables["orders"]
    .join(silver_tables["customers"], on="customer_id", how="left")
    .join(silver_tables["order_items"], on="order_id", how="left")
    .withColumn("receita_item", col("quantity") * parse_currency_br("unit_price"))
    .groupBy("customer_id", "customer_name", "state", "city")
    .agg(
        countDistinct("order_id").alias("total_pedidos"),
        sum("receita_item").alias("total_gasto"),
        avg("receita_item").alias("ticket_medio"),
        min("order_date").alias("primeiro_pedido"),
        max("order_date").alias("ultimo_pedido"),
    )
    .orderBy(col("total_gasto").desc())
)

In [0]:
df_resumo_clientes.write.format("delta").mode("overwrite").saveAsTable("gold.resumo_clientes")
print("✓ gold.resumo_clientes criada")
display(df_resumo_clientes.limit(20))

print("\nCamada Gold concluída!.")